# Finlora Fraud Risk-Scoring: Train / Validation / Test Split

Third deliverable in the pipeline: split the cleaned/merged dataset
(`Data/finlora_cleaned_merged.csv`, produced by `notebooks/cleaned_data.ipynb`)
into train, validation, and test sets, and save each to its own CSV for
downstream notebooks (modeling, evaluation) to consume directly -- so every
notebook that trains or evaluates a model reads the exact same split instead
of re-splitting the data itself.

**Contents**
1. Load data
2. Split: train / validation / test
3. Verify the split
4. Save each set


In [1]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)

## 1. Load data

In [2]:
df = pd.read_csv("../Data/finlora_cleaned_merged.csv")
print("shape:", df.shape)
print(f"overall fraud rate: {df['is_fraud'].mean():.4%}")
df.head()

shape: (126000, 28)
overall fraud rate: 2.6603%


,transaction_id,account_id,account_type,kyc_tier,timestamp,day_of_week,hour_of_day,description,merchant_name,merchant_category,channel,amount,currency,amount_to_avg_ratio,avg_transaction_amount_30d,transaction_velocity_1h,transaction_country,home_country,is_cross_border,device_id,is_new_device,account_age_days,status,is_fraud,account_holder_name,account_created_date,personal_spend_baseline_usd,has_invalid_amount
0,FLR250216186265,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-02-16 14:55:35,Sunday,14,CNP PURCHASE - BOLT,Bolt,Travel,Card Not Present,84233.00,NGN,9.21,9143.33,0,NG,NG,0,NoDevice,-1,903,Completed,0,Amaka Okafor,2022-08-28,63.23,False
1,FLR250423143305,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-23 13:35:40,Wednesday,13,WEB PURCHASE - SLACK,Slack,Subscription/SaaS,Web Dashboard,9143.33,NGN,1.00,9143.33,0,NG,NG,0,NoDevice,-1,969,Declined,0,Amaka Okafor,2022-08-28,63.23,False
2,FLR250424154897,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-24 12:30:34,Thursday,12,MOBILE PURCHASE - JUSTRITE SUPERSTORE,Justrite Superstore,Groceries,Mobile App,20639.00,NGN,2.26,9143.33,0,NG,NG,0,DEV-9055235108,0,970,Completed,0,Amaka Okafor,2022-08-28,63.23,False
3,FLR250428101772,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-28 14:41:59,Monday,14,CNP PURCHASE - ADOBE CREATIVE CLOUD,Adobe Creative Cloud,Subscription/SaaS,Card Not Present,3177.43,NGN,0.21,14891.16,0,NG,NG,0,NoDevice,-1,974,Completed,0,Amaka Okafor,2022-08-28,63.23,False
4,FLR250628102827,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-06-28 16:15:47,Saturday,16,MOBILE PURCHASE - APPLE STORE,Apple Store,Electronics,Mobile App,97839.52,NGN,1.97,49609.59,0,NG,NG,0,DEV-9055235108,0,1035,Declined,0,Amaka Okafor,2022-08-28,63.23,False


## 2. Split: train / validation / test

A **70% / 15% / 15%** split, stratified on `is_fraud` at every step so the rare
positive class (2.66% of rows) keeps the same rate in all three sets. Done as
two sequential splits: first carve off the 15% test set, then split the
remaining 85% into train and validation.

- **train** (70%): fits model parameters.
- **validation** (15%): tunes hyperparameters / picks a decision threshold, without touching the test set.
- **test** (15%): touched exactly once, at the end, for a final unbiased read of model performance.

In [3]:
train_val, test = train_test_split(
    df, test_size=0.15, stratify=df["is_fraud"], random_state=RANDOM_STATE
)
train, val = train_test_split(
    train_val, test_size=0.15 / 0.85, stratify=train_val["is_fraud"], random_state=RANDOM_STATE
)

print("train shape:", train.shape)
print("val shape:  ", val.shape)
print("test shape: ", test.shape)

train shape: (88200, 28)
val shape:   (18900, 28)
test shape:  (18900, 28)


## 3. Verify the split

In [4]:
summary = pd.DataFrame({
    "rows": [len(train), len(val), len(test)],
    "share_of_total": [len(train) / len(df), len(val) / len(df), len(test) / len(df)],
    "fraud_rate": [train["is_fraud"].mean(), val["is_fraud"].mean(), test["is_fraud"].mean()],
}, index=["train", "validation", "test"])
summary

,rows,share_of_total,fraud_rate
train,88200,0.70,0.026599
validation,18900,0.15,0.026614
test,18900,0.15,0.026614


In [5]:
assert len(train) + len(val) + len(test) == len(df), "rows lost or duplicated across the split"
assert set(train["transaction_id"]) & set(val["transaction_id"]) == set(), "train/val overlap"
assert set(train["transaction_id"]) & set(test["transaction_id"]) == set(), "train/test overlap"
assert set(val["transaction_id"]) & set(test["transaction_id"]) == set(), "val/test overlap"
print("No row loss, no overlap between splits.")

No row loss, no overlap between splits.


**Finding:** all three sets land within a few hundredths of a percentage
point of the overall 2.66% fraud rate, and the three sets are mutually
exclusive with no `transaction_id` shared between any pair -- the stratified
split preserved class balance without leaking rows across sets.

## 4. Save each set

Written to `Data/splits/` as `train.csv`, `val.csv`, and `test.csv`, alongside
the raw and cleaned data already in `Data/`.

In [6]:
OUT_DIR = Path("../Data/splits")
OUT_DIR.mkdir(parents=True, exist_ok=True)

train.to_csv(OUT_DIR / "train.csv", index=False)
val.to_csv(OUT_DIR / "val.csv", index=False)
test.to_csv(OUT_DIR / "test.csv", index=False)

print("Saved:")
for name, split in [("train.csv", train), ("val.csv", val), ("test.csv", test)]:
    print(f"  {OUT_DIR / name}  ({len(split):,} rows)")

Saved:
  ..\Data\splits\train.csv  (88,200 rows)
  ..\Data\splits\val.csv  (18,900 rows)
  ..\Data\splits\test.csv  (18,900 rows)
